# Modello MOIRAI (Zero-Shot Foundation Model) - Bonds & Macro data

Testiamo **MOIRAI** (Masked Encoder-based Universal Time Series Representation Learning) sui dati combinati Macro-Bond. MOIRAI è un *Foundation Model* sviluppato da Salesforce per le serie storiche, che effettua inferenza *zero-shot*.

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

# Importiamo Moirai e GluonTS
from gluonts.dataset.pandas import PandasDataset
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

print("Librerie per MOIRAI caricate correttamente.")


/home/francesco/intelligent_systems/progetto/bond_screener/moirari_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerie per MOIRAI caricate correttamente.


Prepariamo il dataset. Poiché abbiamo molti bond (molti `isincode`), dovremo preparare un dataset GluonTS indicando `isincode` come identificativo (item_id) in un `PandasDataset`.

In [ ]:
# Carichiamo i dati
df_moirai = pd.read_csv('./data/df_bond_macro.csv', index_col=0).reset_index()

if 'index' in df_moirai.columns:
    df_moirai = df_moirai.rename(columns={'index': 'isincode'})

df_moirai['referencedate'] = pd.to_datetime(df_moirai['referencedate'])
df_moirai = df_moirai.sort_values(['referencedate', 'isincode'])

FORECAST_HORIZON = 90
target_col = 'pricevalue'

print("Inizializzazione del modello MOIRAI...")
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small"),
    prediction_length=FORECAST_HORIZON,
    context_length=300, 
    patch_size="auto",
    num_samples=100, 
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)
print("Modello MOIRAI inizializzato.")


Inizializzazione del modello MOIRAI...


Modello MOIRAI inizializzato.


Eseguiamo l'inferenza. Tratteremo alcuni ISIN di test nei periodi di test stabiliti dalla split time-series. Data la pesantezza computazionale di valutare centinaia di bond con un foundation model su CPU, campioneremo un sottoinsieme di ISIN e punti nel tempo.

In [ ]:
unique_dates = np.sort(df_moirai['referencedate'].unique())
n_splits = 5
test_size_days = len(unique_dates) // (n_splits + 2)
gap_days = FORECAST_HORIZON

moirai_roc_auc_scores = []
moirai_f1_scores = []

# Campioniamo solo alcuni ISIN per ridurre i tempi di inferenza in questo notebook di esempio
np.random.seed(42)
all_isins = df_moirai['isincode'].unique()
sample_isins = np.random.choice(all_isins, min(40, len(all_isins)), replace=False)
print(f"Valutiamo MOIRAI su un sample di {len(sample_isins)} ISIN per velocizzare il processo.")

df_sample = df_moirai[df_moirai['isincode'].isin(sample_isins)].copy()
df_sample = df_sample.set_index('referencedate')

for fold in range(n_splits):
    print(f"\n--- Fold {fold+1} ---")
    
    test_start_idx = len(unique_dates) - test_size_days * (n_splits - fold)
    test_end_idx = test_start_idx + test_size_days
    
    test_dates = unique_dates[test_start_idx:test_end_idx]
    
    # Prendiamo 3 punti chiave nel periodo di test da usare come 'cutoff' (il momento in cui facciamo la previsione)
    # Assicuriamoci che ci sia abbastanza spazio futuro (FORECAST_HORIZON) dopo il cutoff
    valid_test_dates = test_dates[:-FORECAST_HORIZON]
    if len(valid_test_dates) < 3:
        valid_test_dates = test_dates # Fallback se il set è troppo piccolo
    
    cutoff_dates = valid_test_dates[::max(1, len(valid_test_dates)//3)][:3]
    
    y_preds_prob = []
    y_trues = []
    
    for cutoff_date in cutoff_dates:
        # Troviamo la data di target vera a +90 giorni (approx)
        future_idx = np.where(unique_dates == cutoff_date)[0][0] + FORECAST_HORIZON
        if future_idx >= len(unique_dates):
            continue
        future_date = unique_dates[future_idx]
        
        for isin in sample_isins:
            df_isin = df_sample[df_sample['isincode'] == isin]
            
            # Storia fino al cutoff
            history_df = df_isin[df_isin.index <= cutoff_date]
            if len(history_df) < 50:
                continue # Poca storia
                
            # Verifica se abbiamo il dato futuro
            future_data = df_isin[df_isin.index == future_date]
            if future_data.empty:
                continue
                
            current_val = history_df.iloc[-1][target_col]
            future_val = future_data.iloc[0][target_col]
            
            true_target = 1 if future_val > current_val else 0
            
            # Prepariamo per GluonTS. Moirai richiede una frequenza ('B' per business days è l'ideale per i bond)
            hist_ts = history_df[[target_col]].copy()
            hist_ts = hist_ts[~hist_ts.index.duplicated(keep='last')]
            hist_ts = hist_ts.asfreq('B').ffill()
            
            ds = PandasDataset(hist_ts, target=target_col)
            predictor = model.create_predictor(batch_size=1, device="cpu")
            
            try:
                forecast_it = predictor.predict(ds)
                forecast = next(forecast_it)
                pred_90d = forecast.samples[:, -1]
                prob_up = np.mean(pred_90d > current_val)
                
                y_preds_prob.append(prob_up)
                y_trues.append(true_target)
            except Exception as e:
                pass # Ignoriamo se ci sono errori di formattazione temporale per un bond specifico
                
    if len(y_trues) == 0:
        print("Nessuna previsione valida in questo fold.")
        continue
        
    roc = roc_auc_score(y_trues, y_preds_prob)
    y_preds_class = [1 if p > 0.5 else 0 for p in y_preds_prob]
    f1 = f1_score(y_trues, y_preds_class)
    
    print(f"Fold ROC-AUC: {roc:.3f} | F1: {f1:.3f} ")
    
    moirai_roc_auc_scores.append(roc)
    moirai_f1_scores.append(f1)

print(f"\n=======================================================")
print(f"Risultato Finale MOIRAI (Media su {len(moirai_roc_auc_scores)} folds):")
if moirai_roc_auc_scores:
    print(f"Mean ROC-AUC: {np.mean(moirai_roc_auc_scores):.3f}")
    print(f"Mean F1-Score: {np.mean(moirai_f1_scores):.3f}")
else:
    print("Nessun punteggio calcolato.")
print(f"=======================================================")


Valutiamo MOIRAI su un sample di 40 ISIN per velocizzare il processo.

--- Fold 1 ---
Fold ROC-AUC: 0.448 | F1: 0.203 (Soglia dinamica: 0.585)

--- Fold 2 ---
Fold ROC-AUC: 0.022 | F1: 0.652 (Soglia dinamica: 0.250)

--- Fold 3 ---
Fold ROC-AUC: 0.798 | F1: 0.659 (Soglia dinamica: 0.650)

--- Fold 4 ---
Fold ROC-AUC: 0.681 | F1: 0.578 (Soglia dinamica: 0.560)

--- Fold 5 ---
Fold ROC-AUC: 0.743 | F1: 0.211 (Soglia dinamica: 0.420)

Risultato Finale MOIRAI (Media su 5 folds):
Mean ROC-AUC: 0.539
Mean F1-Score: 0.461
